# 02 - Data Cleaning

## 🎯 Objectif
Nettoyer les données brutes pour préparer le **Feature Engineering**

## 📋 Stratégie
Basé sur l'EDA, nous allons :
1. **Parser les colonnes complexes** (nutrition, tags, ingredients, steps)
2. **Nettoyer les valeurs aberrantes** (outliers, missing values)
3. **Filtrer les données invalides** (duplicatas, valeurs impossibles)
4. **Exporter les fichiers cleaned** prêts pour le feature engineering

## 🔧 Actions :
1. Charger les données brutes (interactions, recipes, users)
2. Parser nutrition (liste → 7 colonnes float)
3. Parser tags (liste → 7 colonnes binaires)
4. Imputer valeurs manquantes (médiane)
5. Winsorizer outliers (99e/1e percentile)
6. Filtrer recettes/interactions invalides
7. Export : `recipes_clean.csv`, `interactions_clean.csv`

## 📦 Outputs :
- **`recipes_clean.csv`** : Recettes nettoyées avec nutrition + tags parsés
- **`interactions_clean.csv`** : Interactions filtrées (ratings valides)


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style("whitegrid")
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

# Helper pour parser les colonnes liste
def parse_list_col(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x if isinstance(x, list) else []

print('✅ Configuration OK')
print(f'  Data raw: {DATA_RAW}')
print(f'  Data processed: {DATA_PROCESSED}')


✅ Configuration OK
  Data raw: ../data/raw
  Data processed: ../data/processed


In [6]:
print("=" * 80)
print("CHARGEMENT DES DONNÉES")
print("=" * 80)

# Charger les 3 fichiers principaux
interactions = pd.read_csv(DATA_RAW / 'interactions_train.csv')
recipes = pd.read_csv(DATA_RAW / 'RAW_recipes.csv')
users = pd.read_csv(DATA_RAW / 'PP_users.csv')

print(f'\n DONNÉES CHARGÉES:')
print(f'   • Interactions: {interactions.shape}')
print(f'   • Recipes:      {recipes.shape}')
print(f'   • Recipes list:      {recipes.columns.tolist()}')
print(f'   • Users:        {users.shape}')

print(f'\n UTILISATEURS UNIQUES:')
print(f'   • Dans interactions: {interactions["user_id"].nunique():,}')
print(f'   • Dans users.csv:    {users["u"].nunique():,}')

print(f'\n RECETTES UNIQUES:')
print(f'   • Dans interactions: {interactions["recipe_id"].nunique():,}')
print(f'   • Dans recipes.csv:  {recipes["id"].nunique():,}')


CHARGEMENT DES DONNÉES



 DONNÉES CHARGÉES:
   • Interactions: (698901, 6)
   • Recipes:      (231637, 12)
   • Recipes list:      ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
   • Users:        (25076, 6)

 UTILISATEURS UNIQUES:
   • Dans interactions: 25,076
   • Dans users.csv:    25,076

 RECETTES UNIQUES:
   • Dans interactions: 160,901
   • Dans recipes.csv:  231,637


In [7]:
print("=" * 80)
print("ÉTAPE 1 - PARSER NUTRITION ET TAGS DES RECETTES")
print("=" * 80)

# 1. Parser la colonne nutrition (7 valeurs)
recipes['nutrition_parsed'] = recipes['nutrition'].apply(parse_list_col)

recipes['calories'] = recipes['nutrition_parsed'].apply(lambda x: x[0] if len(x) > 0 else np.nan)
recipes['total_fat'] = recipes['nutrition_parsed'].apply(lambda x: x[1] if len(x) > 1 else np.nan)
recipes['sugar'] = recipes['nutrition_parsed'].apply(lambda x: x[2] if len(x) > 2 else np.nan)
recipes['sodium'] = recipes['nutrition_parsed'].apply(lambda x: x[3] if len(x) > 3 else np.nan)
recipes['protein'] = recipes['nutrition_parsed'].apply(lambda x: x[4] if len(x) > 4 else np.nan)
recipes['saturated_fat'] = recipes['nutrition_parsed'].apply(lambda x: x[5] if len(x) > 5 else np.nan)
recipes['carbohydrates'] = recipes['nutrition_parsed'].apply(lambda x: x[6] if len(x) > 6 else np.nan)

nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates']

print(f'\n✅ Nutrition parsée (7 colonnes)')

# 2. Parser les tags et créer 7 catégories binaires (basées sur EDA)
recipes['tags_parsed'] = recipes['tags'].apply(parse_list_col)

recipes['tag_time_to_make'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['time-to-make', '60-minutes-or-less', '30-minutes-or-less', '15-minutes-or-less']) else 0
)
recipes['tag_course'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['course', 'main-dish', 'desserts', 'side-dishes', 'appetizers']) else 0
)
recipes['tag_main_ingredient'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['main-ingredient', 'meat', 'vegetables', 'fruit', 'poultry']) else 0
)
recipes['tag_dietary'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['dietary', 'low-sodium', 'low-carb', 'healthy', 'vegetarian']) else 0
)
recipes['tag_easy'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['easy', 'beginner-cook', '3-steps-or-less', '5-ingredients-or-less']) else 0
)
recipes['tag_occasion'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['occasion', 'holiday-event', 'dinner-party', 'kid-friendly']) else 0
)
recipes['tag_cuisine'] = recipes['tags_parsed'].apply(
    lambda t: 1 if any(k in t for k in ['cuisine', 'north-american', 'american', 'european', 'asian']) else 0
)

tag_cols = ['tag_time_to_make', 'tag_course', 'tag_main_ingredient', 'tag_dietary', 'tag_easy', 'tag_occasion', 'tag_cuisine']

print(f'\n✅ Tags parsés (7 catégories binaires)')
for col in tag_cols:
    pct = 100 * recipes[col].sum() / len(recipes)
    print(f'   • {col}: {pct:.1f}%')

print(f'\n RECETTES ENRICHIES:')
print(f'   • Shape: {recipes.shape}')
print(f'   • Nouvelles colonnes: {len(nutrition_cols) + len(tag_cols)} (7 nutrition + 7 tags)')


ÉTAPE 1 - PARSER NUTRITION ET TAGS DES RECETTES

✅ Nutrition parsée (7 colonnes)

✅ Tags parsés (7 catégories binaires)
   • tag_time_to_make: 97.3%
   • tag_course: 94.2%
   • tag_main_ingredient: 73.6%
   • tag_dietary: 71.3%
   • tag_easy: 54.4%
   • tag_occasion: 51.4%
   • tag_cuisine: 39.4%

 RECETTES ENRICHIES:
   • Shape: (231637, 28)
   • Nouvelles colonnes: 14 (7 nutrition + 7 tags)


## ÉTAPE 2 - Nettoyer les Recettes (Outliers & Missing Values)

In [8]:
print("=" * 80)
print("NETTOYAGE DES RECETTES - OUTLIERS & MISSING VALUES")
print("=" * 80)

# 1. Imputer valeurs manquantes (médiane)
print(f'\n IMPUTATION (médiane):')
for col in nutrition_cols:
    n_missing = recipes[col].isnull().sum()
    if n_missing > 0:
        median = recipes[col].median()
        recipes[col].fillna(median, inplace=True)
        print(f'   • {col}: {n_missing:,} imputés → médiane={median:.1f}')

# 2. Winsorization (capper les outliers au 99e percentile)
print(f'\n  WINSORIZATION (99e/1e percentile):')
total_outliers = 0
for col in nutrition_cols + ['minutes']:
    p99, p1 = recipes[col].quantile([0.99, 0.01])
    n_above = (recipes[col] > p99).sum()
    n_below = (recipes[col] < p1).sum()
    total_outliers += n_above + n_below
    recipes.loc[recipes[col] > p99, col] = p99
    recipes.loc[recipes[col] < p1, col] = p1
    if n_above + n_below > 0:
        print(f'   • {col}: {n_above + n_below:,} cappés (p1={p1:.1f}, p99={p99:.1f})')

print(f'\n   Total outliers cappés: {total_outliers:,}')

# 3. Filtrer recettes invalides
print(f'\n🧹 FILTRAGE RECETTES INVALIDES:')
print(f'   Avant: {len(recipes):,}')

recipes_clean = recipes.copy()
recipes_clean = recipes_clean.dropna(subset=['id', 'name'])
recipes_clean = recipes_clean.drop_duplicates(subset=['id'])
recipes_clean = recipes_clean[(recipes_clean['minutes'] > 0) & (recipes_clean['minutes'] <= 1440)]
recipes_clean = recipes_clean[recipes_clean['calories'] > 0]
recipes_clean = recipes_clean[(recipes_clean['n_steps'] > 0) & (recipes_clean['n_ingredients'] > 0)]

print(f'   Après: {len(recipes_clean):,}')
print(f'   Supprimées: {len(recipes) - len(recipes_clean):,} ({100*(len(recipes)-len(recipes_clean))/len(recipes):.1f}%)')

print(f'\n✅ Recettes nettoyées: {recipes_clean.shape}')


NETTOYAGE DES RECETTES - OUTLIERS & MISSING VALUES

 IMPUTATION (médiane):

  WINSORIZATION (99e/1e percentile):
   • calories: 4,631 cappés (p1=18.4, p99=3517.0)


   • total_fat: 2,314 cappés (p1=0.0, p99=302.0)
   • sugar: 2,317 cappés (p1=0.0, p99=1141.6)
   • sodium: 2,311 cappés (p1=0.0, p99=219.0)
   • protein: 2,295 cappés (p1=0.0, p99=188.0)
   • saturated_fat: 2,312 cappés (p1=0.0, p99=404.0)
   • carbohydrates: 2,315 cappés (p1=0.0, p99=154.0)
   • minutes: 3,953 cappés (p1=2.0, p99=903.9)

   Total outliers cappés: 22,448

🧹 FILTRAGE RECETTES INVALIDES:
   Avant: 231,637
   Après: 231,635
   Supprimées: 2 (0.0%)

✅ Recettes nettoyées: (231635, 28)


## ÉTAPE 3 - Nettoyer les Interactions

In [9]:
print("=" * 80)
print("NETTOYAGE DES INTERACTIONS")
print("=" * 80)

# Filtrer les interactions invalides
print(f'\n FILTRAGE INTERACTIONS INVALIDES:')
print(f'   Avant: {len(interactions):,}')

interactions_clean = interactions.copy()
# Garder seulement ratings valides (0-5)
interactions_clean = interactions_clean[interactions_clean['rating'].between(0, 5)]
# Supprimer duplicatas
interactions_clean = interactions_clean.drop_duplicates()
# Supprimer valeurs manquantes critiques
interactions_clean = interactions_clean.dropna(subset=['user_id', 'recipe_id', 'rating'])

print(f'   Après: {len(interactions_clean):,}')
print(f'   Supprimées: {len(interactions) - len(interactions_clean):,} ({100*(len(interactions)-len(interactions_clean))/len(interactions):.1f}%)')

print(f'\n✅ Interactions nettoyées: {interactions_clean.shape}')


NETTOYAGE DES INTERACTIONS

 FILTRAGE INTERACTIONS INVALIDES:
   Avant: 698,901
   Après: 698,901
   Supprimées: 0 (0.0%)

✅ Interactions nettoyées: (698901, 6)


## ÉTAPE 4 - Export des Fichiers Cleaned

In [12]:
print("=" * 80)
print("EXPORT DES FICHIERS CLEANED")
print("=" * 80)

# Sélectionner les colonnes à exporter pour recipes
recipes_export_cols = [
    'id', 'name', 'minutes', 'n_steps', 'n_ingredients',
    'calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates',
    'tag_time_to_make', 'tag_course', 'tag_main_ingredient', 'tag_dietary', 'tag_easy', 'tag_occasion', 'tag_cuisine'
]

# Export recipes_clean.csv
recipes_output = DATA_PROCESSED / 'recipes_clean.csv'
recipes_clean[recipes_export_cols].to_csv(recipes_output, index=False)

print(f'\n✅ FICHIER 1 - RECIPES_CLEAN.CSV:')
print(f'   • Chemin: {recipes_output}')
print(f'   • Shape: {recipes_clean[recipes_export_cols].shape}')
print(f'   • Taille: {recipes_output.stat().st_size / 1024 / 1024:.2f} MB')

print(f'   • Colonnes: {len(recipes_export_cols)}')

print(f'\n✅ Fichiers prêts pour Feature Engineering!')

# Export interactions_clean.csv

interactions_output = DATA_PROCESSED / 'interactions_clean.csv'
print(f'   • Colonnes: {len(interactions_clean.columns)}')

interactions_clean.to_csv(interactions_output, index=False)
print(f'   • Taille: {interactions_output.stat().st_size / 1024 / 1024:.2f} MB')

print(f'   • Shape: {interactions_clean.shape}')

print(f'\n✅ FICHIER 2 - INTERACTIONS_CLEAN.CSV:')
print(f'   • Chemin: {interactions_output}')

EXPORT DES FICHIERS CLEANED

✅ FICHIER 1 - RECIPES_CLEAN.CSV:
   • Chemin: ../data/processed/recipes_clean.csv
   • Shape: (231635, 19)
   • Taille: 20.82 MB
   • Colonnes: 19

✅ Fichiers prêts pour Feature Engineering!
   • Colonnes: 6
   • Taille: 26.27 MB
   • Shape: (698901, 6)

✅ FICHIER 2 - INTERACTIONS_CLEAN.CSV:
   • Chemin: ../data/processed/interactions_clean.csv


## 📋 Synthèse Finale

In [13]:
print("\n" + "=" * 80)
print(" SYNTHÈSE FINALE - DATA CLEANING")
print("=" * 80)

print(f'\n OBJECTIF ATTEINT:')
print(f'   Nettoyer et préparer les données pour le Feature Engineering')

print(f'\n DONNÉES SOURCES:')
print(f'   • Interactions (train):  {len(interactions):,}')
print(f'   • Recettes (RAW):        {len(recipes):,}')
print(f'   • Users (PP):            {len(users):,}')

print(f'\n TRAITEMENTS RÉALISÉS:')
print(f'   1. ✅ Parser nutrition (7 colonnes float)')
print(f'   2. ✅ Parser tags (7 catégories binaires)')
print(f'   3. ✅ Imputer valeurs manquantes (médiane)')
print(f'   4. ✅ Winsorizer outliers (99e/1e percentile)')
print(f'   5. ✅ Filtrer recettes invalides')
print(f'   6. ✅ Filtrer interactions invalides')

print(f'\n FICHIERS EXPORTÉS:')
print(f'   1. recipes_clean.csv')
print(f'      • Shape: {recipes_clean[recipes_export_cols].shape}')
print(f'      • Colonnes: {len(recipes_export_cols)} (nutrition + tags parsés)')
print(f'   2. interactions_clean.csv')
print(f'      • Shape: {interactions_clean.shape}')
print(f'      • Ratings valides: 0-5')

print(f'\n PROCHAINE ÉTAPE:')
print(f'   ✅ Notebook: 03_feature_engineering.ipynb')
print(f'      → Créer profils utilisateurs (agrégation par user_id)')
print(f'      → Export: users_profiles.csv')
print("=" * 80)



 SYNTHÈSE FINALE - DATA CLEANING

 OBJECTIF ATTEINT:
   Nettoyer et préparer les données pour le Feature Engineering

 DONNÉES SOURCES:
   • Interactions (train):  698,901
   • Recettes (RAW):        231,637
   • Users (PP):            25,076

 TRAITEMENTS RÉALISÉS:
   1. ✅ Parser nutrition (7 colonnes float)
   2. ✅ Parser tags (7 catégories binaires)
   3. ✅ Imputer valeurs manquantes (médiane)
   4. ✅ Winsorizer outliers (99e/1e percentile)
   5. ✅ Filtrer recettes invalides
   6. ✅ Filtrer interactions invalides

 FICHIERS EXPORTÉS:
   1. recipes_clean.csv
      • Shape: (231635, 19)
      • Colonnes: 19 (nutrition + tags parsés)
   2. interactions_clean.csv
      • Shape: (698901, 6)
      • Ratings valides: 0-5

 PROCHAINE ÉTAPE:
   ✅ Notebook: 03_feature_engineering.ipynb
      → Créer profils utilisateurs (agrégation par user_id)
      → Export: users_profiles.csv
